# Install Detectron2

Run this cell ONCE to install Detectron2:

In [3]:
# Install Detectron2
# Adjust the CUDA version based on your system:
# - cu118 for CUDA 11.8
# - cu121 for CUDA 12.1
# - cpu for CPU-only

import sys
import torch

# Check PyTorch and CUDA versions
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")

# Install command - adjust based on your CUDA version
# For CUDA 11.8:
# !pip install detectron2 -f https://dl.fbaipublicfiles.com/detectron2/wheels/cu118/torch2.0/index.html

# For CPU only:
!pip install detectron2 -f https://dl.fbaipublicfiles.com/detectron2/wheels/cpu/torch2.0/index.html

PyTorch: 2.9.1+cpu
CUDA available: False
Looking in links: https://dl.fbaipublicfiles.com/detectron2/wheels/cpu/torch2.0/index.html


ERROR: Could not find a version that satisfies the requirement detectron2 (from versions: none)
ERROR: No matching distribution found for detectron2


In [4]:
import torch
import cv2
import numpy as np
from detectron2.config import get_cfg
from detectron2 import model_zoo
from detectron2.modeling import build_model
from detectron2.checkpoint import DetectionCheckpointer
import detectron2.data.transforms as T

print("✓ All imports successful")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

ModuleNotFoundError: No module named 'detectron2'

In [ ]:
# UPDATE THESE PATHS
weights_path = r"C:\Research\Thermal-Dataset\ExplainableAI\weights\fastercnn\resnet50c4\model_final.pth"
test_image = r"C:\Research\Thermal-Dataset\ExplainableAI\weights\test\weapon115.jpg"

# UPDATE THIS: Your class names (NO background class in Detectron2)
num_classes = 5
class_names = [
    'handgun',
    'knife', 
    'person',
    'revolver',
    'rifle'
]

print(f"Weights: {weights_path}")
print(f"Test image: {test_image}")
print(f"Classes: {class_names}")

In [ ]:
# Setup Detectron2 config
cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file("COCO-Detection/faster_rcnn_R_50_C4_1x.yaml"))

# Model settings
cfg.MODEL.ROI_HEADS.NUM_CLASSES = num_classes
cfg.MODEL.WEIGHTS = weights_path
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5  # Confidence threshold
cfg.MODEL.DEVICE = str(device)

# Build model
model = build_model(cfg)
model.eval()

# Load checkpoint
checkpointer = DetectionCheckpointer(model)
checkpointer.load(weights_path)

print("✓ Model loaded successfully!")
print(f"Model type: {type(model)}")

In [ ]:
# Load and preprocess image
img = cv2.imread(test_image)
print(f"Image shape: {img.shape}")

# Detectron2 expects BGR format (OpenCV default)
height, width = img.shape[:2]

# Prepare input
aug = T.ResizeShortestEdge([800, 800], 1333)
image_tensor = aug.get_transform(img).apply_image(img)
image_tensor = torch.as_tensor(image_tensor.astype("float32").transpose(2, 0, 1))

inputs = [{"image": image_tensor, "height": height, "width": width}]

print(f"Input tensor shape: {image_tensor.shape}")
print("✓ Image preprocessed")

In [ ]:
# Run detection
with torch.no_grad():
    outputs = model(inputs)

# Extract results
instances = outputs[0]["instances"].to("cpu")
boxes = instances.pred_boxes.tensor.numpy()
scores = instances.scores.numpy()
classes = instances.pred_classes.numpy()

print(f"\n✓ Found {len(boxes)} detections:")
for i, (box, score, cls) in enumerate(zip(boxes, scores, classes)):
    print(f"  {i+1}. {class_names[cls]}: {score:.3f} at {box.astype(int)}")

In [ ]:
# Visualize detections
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
img_vis = img_rgb.copy()

for box, score, cls in zip(boxes, scores, classes):
    x1, y1, x2, y2 = box.astype(int)
    cv2.rectangle(img_vis, (x1, y1), (x2, y2), (0, 255, 0), 3)
    label = f"{class_names[cls]}: {score:.2f}"
    cv2.putText(img_vis, label, (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

cv2.imwrite('detection_result.png', cv2.cvtColor(img_vis, cv2.COLOR_RGB2BGR))
print("✓ Saved detection_result.png")

from IPython.display import Image, display
display(Image('detection_result.png'))

## Grad-CAM for Detectron2 Faster R-CNN

In [ ]:
def generate_gradcam_detectron2(model, image_tensor, detections):
    """
    Generate Grad-CAM for Detectron2 Faster R-CNN
    """
    print("\n" + "="*60)
    print("GENERATING GRAD-CAM")
    print("="*60)
    
    if len(detections['boxes']) == 0:
        print("⚠ No detections found")
        return None
    
    # Get backbone (ResNet50-C4)
    backbone = model.backbone
    
    # Find last conv layer in res5 (C4)
    target_layer = backbone.res5[-1].conv3
    print(f"Target layer: res5 block 2, conv3")
    
    # Storage
    activations = []
    gradients = []
    
    def forward_hook(module, input, output):
        activations.append(output.detach().cpu())
    
    def backward_hook(module, grad_input, grad_output):
        if grad_output[0] is not None:
            gradients.append(grad_output[0].detach().cpu())
    
    # Register hooks
    h1 = target_layer.register_forward_hook(forward_hook)
    h2 = target_layer.register_full_backward_hook(backward_hook)
    
    try:
        # Forward pass with gradients
        model.train()  # Enable gradients
        model.zero_grad()
        
        # Prepare input that requires gradients
        img_input = image_tensor.clone().requires_grad_(True)
        inputs = [{"image": img_input, "height": detections['height'], "width": detections['width']}]
        
        # Forward
        outputs = model(inputs)
        
        # Get highest confidence score
        instances = outputs[0]["instances"]
        if len(instances) > 0:
            max_score = instances.scores.max()
            print(f"Max confidence: {max_score:.3f}")
            
            # Backward
            max_score.backward()
            print("✓ Gradients computed")
        else:
            h1.remove()
            h2.remove()
            model.eval()
            return None
            
    except Exception as e:
        print(f"✗ Error: {e}")
        h1.remove()
        h2.remove()
        model.eval()
        return None
    
    # Remove hooks
    h1.remove()
    h2.remove()
    model.eval()
    
    # Generate CAM
    if len(gradients) > 0 and len(activations) > 0:
        grads = gradients[0].numpy()[0]
        acts = activations[0].numpy()[0]
        
        # Global average pooling
        weights = np.mean(grads, axis=(1, 2))
        
        # Weighted sum
        cam = np.zeros(acts.shape[1:], dtype=np.float32)
        for i, w in enumerate(weights):
            cam += w * acts[i]
        
        # ReLU + Normalize
        cam = np.maximum(cam, 0)
        if cam.max() > 0:
            cam = cam / cam.max()
        
        print(f"✓ CAM generated: {cam.shape}")
        return cam
    else:
        print("✗ No gradients captured")
        return None

print("✓ Grad-CAM function defined")

In [ ]:
# Generate Grad-CAM
detections_dict = {
    'boxes': boxes,
    'scores': scores,
    'classes': classes,
    'height': height,
    'width': width
}

cam = generate_gradcam_detectron2(model, image_tensor, detections_dict)

In [ ]:
# Create XAI visualization
img_resized = cv2.resize(img_rgb, (640, 640))

# Resize CAM
if cam is not None:
    cam_resized = cv2.resize(cam, (640, 640))
    heatmap = cv2.applyColorMap(np.uint8(255 * cam_resized), cv2.COLORMAP_JET)
    heatmap_rgb = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
else:
    heatmap_rgb = np.zeros_like(img_resized)

# Scale boxes for 640x640 visualization
scale_x = 640 / width
scale_y = 640 / height
boxes_scaled = boxes.copy()
boxes_scaled[:, [0, 2]] *= scale_x
boxes_scaled[:, [1, 3]] *= scale_y

# Panel 1: Original
panel1 = img_resized.copy()
cv2.putText(panel1, '1. Original Image', (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)

# Panel 2: Detections
panel2 = img_resized.copy()
for box, score, cls in zip(boxes_scaled, scores, classes):
    x1, y1, x2, y2 = box.astype(int)
    cv2.rectangle(panel2, (x1, y1), (x2, y2), (0, 255, 0), 3)
    label = f'{class_names[cls]}: {score:.2f}'
    cv2.putText(panel2, label, (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
cv2.putText(panel2, f'2. Detections ({len(boxes)})', (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)

# Panel 3: Grad-CAM
panel3 = heatmap_rgb.copy()
cv2.putText(panel3, '3. Grad-CAM', (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
cv2.putText(panel3, 'Red = High Attention', (10, 620), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

# Panel 4: Overlay
if cam is not None:
    panel4 = cv2.addWeighted(img_resized, 0.5, heatmap_rgb, 0.5, 0)
    for box in boxes_scaled:
        x1, y1, x2, y2 = box.astype(int)
        cv2.rectangle(panel4, (x1, y1), (x2, y2), (255, 255, 255), 3)
else:
    panel4 = img_resized.copy()
cv2.putText(panel4, '4. Explanation', (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)

# Combine
top_row = np.hstack([panel1, panel2])
bottom_row = np.hstack([panel3, panel4])
final_viz = np.vstack([top_row, bottom_row])

cv2.imwrite('xai_result.png', cv2.cvtColor(final_viz, cv2.COLOR_RGB2BGR))
print("✓ Saved xai_result.png")

display(Image('xai_result.png'))